# Experiment 3: Data Cleaning & Storage (News API)
Collect news, preprocess text, perform sentiment analysis, and store results.

In [1]:
# Step 1: Install (run if needed)
!pip install newsapi-python pandas nltk pymongo

In [2]:
# Step 2: Import Libraries
from newsapi import NewsApiClient
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from pymongo import MongoClient
import random

nltk.download('stopwords', quiet=True)
print('Libraries loaded')

Libraries loaded


In [3]:
# Step 3: Collect News
API_KEY = 'YOUR_API_KEY'
newsapi = NewsApiClient(api_key=API_KEY)

query = 'Apple OR Samsung OR smartphone'

sample_articles = [
    {
        'publishedAt': '2026-05-01T10:00:00Z',
        'title': 'Apple launches new smartphone features',
        'description': 'The latest update improves battery life and camera quality.',
        'content': 'Apple introduces amazing new smartphone features with excellent performance and great battery life.',
        'url': 'https://example.com/apple-launch'
    },
    {
        'publishedAt': '2026-05-02T12:30:00Z',
        'title': 'Samsung faces supply chain issues',
        'description': 'Production delays may affect device availability.',
        'content': 'Samsung reports a crisis in the supply chain causing poor availability and potential loss in revenue.',
        'url': 'https://example.com/samsung-issues'
    },
    {
        'publishedAt': '2026-05-03T09:15:00Z',
        'title': 'Smartphone market remains stable',
        'description': 'Analysts expect steady demand over the next quarter.',
        'content': 'The smartphone market shows neutral trends with balanced demand and moderate success.',
        'url': 'https://example.com/market-stable'
    }
]

try:
    if API_KEY == 'YOUR_API_KEY' or not API_KEY.strip():
        raise ValueError('No valid News API key provided')
    response = newsapi.get_everything(q=query, language='en', sort_by='relevancy', page_size=50)
    articles = response['articles']
    print('Articles fetched:', len(articles))
except Exception as e:
    print('Using sample news data:', e)
    articles = sample_articles
    print('Articles fetched:', len(articles))


df = pd.DataFrame(articles)
df = df[['publishedAt','title','description','content','url']].copy()

df['Text'] = df['title'].fillna('') + ' ' + df['description'].fillna('')
df.rename(columns={'publishedAt':'Date'}, inplace=True)

df['Likes'] = [random.randint(10,500) for _ in range(len(df))]
df['Shares'] = [random.randint(5,200) for _ in range(len(df))]

df[['Date','Text','Likes','Shares']].head()

Using sample news data: No valid News API key provided
Articles fetched: 3


,Date,Text,Likes,Shares
0,2026-05-01T10:00:00Z,Apple launches new smartphone features The lat...,37,153
1,2026-05-02T12:30:00Z,Samsung faces supply chain issues Production d...,433,70
2,2026-05-03T09:15:00Z,Smartphone market remains stable Analysts expe...,467,61


In [4]:
# Step 4: Text Preprocessing
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['Cleaned_Text'] = df['Text'].apply(clean_text)

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    return ' '.join([w for w in text.split() if w not in stop_words])

df['Processed_Text'] = df['Cleaned_Text'].apply(remove_stopwords)

df[['Text','Processed_Text']].head()

,Text,Processed_Text
0,Apple launches new smartphone features The lat...,apple launches new smartphone features latest ...
1,Samsung faces supply chain issues Production d...,samsung faces supply chain issues production d...
2,Smartphone market remains stable Analysts expe...,smartphone market remains stable analysts expe...


In [5]:
# Step 5: Sentiment Analysis
positive_words = {'good','great','love','excellent','amazing','happy','success','win'}
negative_words = {'bad','worst','hate','poor','terrible','fail','loss','crisis'}

def get_sentiment(text):
    words = set(text.split())
    if words & positive_words:
        return 'Positive'
    elif words & negative_words:
        return 'Negative'
    return 'Neutral'

df['Sentiment'] = df['Processed_Text'].apply(get_sentiment)
df[['Processed_Text','Sentiment']].head(10)

,Processed_Text,Sentiment
0,apple launches new smartphone features latest ...,Neutral
1,samsung faces supply chain issues production d...,Neutral
2,smartphone market remains stable analysts expe...,Neutral


In [6]:
# Step 6: Filter Data
negative = df[df['Sentiment']=='Negative']
positive = df[df['Sentiment']=='Positive']

print('Negative:', len(negative))
print('Positive:', len(positive))
print('\nDistribution:\n', df['Sentiment'].value_counts())

Negative: 0
Positive: 0

Distribution:
 Sentiment
Neutral    3
Name: count, dtype: int64


In [7]:
# Step 7: Save Data
df.to_csv('news_articles.csv', index=False)
print('Saved CSV')

try:
    client = MongoClient('mongodb://localhost:27017/', serverSelectionTimeoutMS=3000)
    db = client['SocialMediaDB']
    col = db['NewsArticles']
    col.insert_many(df.to_dict('records'))
    print('Stored in MongoDB')
except Exception as e:
    print('MongoDB not connected:', e)

Saved CSV
MongoDB not connected: localhost:27017: [Errno 111] Connection refused (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 3.0s, Topology Description: <TopologyDescription id: 69f7b793ff77331fef255ddc, topology_type: Unknown, servers: [<ServerDescription ('localhost', 27017) server_type: Unknown, rtt: None, error=AutoReconnect('localhost:27017: [Errno 111] Connection refused (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms)')>]>


In [8]:
# Step 8: Summary
total = len(df)
pos = len(df[df['Sentiment']=='Positive'])
neg = len(df[df['Sentiment']=='Negative'])
neu = len(df[df['Sentiment']=='Neutral'])

print('Total:', total)
print('Positive:', pos)
print('Negative:', neg)
print('Neutral:', neu)

Total: 3
Positive: 0
Negative: 0
Neutral: 3
